# 01 — Boosting as fitting residuals (by hand)

*Chapter 08 · Gradient Boosting · Notebook 1 of 6*

Welcome to gradient boosting — the general form of the boosting family, and one of the most
effective methods for tabular data. We begin the way we began AdaBoost: by building the core idea
with our own hands, before any library call.

The idea is short. Start with a rough model. Look at what it gets wrong — the **residuals**. Train a
small tree to predict those residuals, and add it on. Repeat. Each new tree cleans up what the
previous model left behind.

**Prerequisites.** Chapter 07 (boosting trains weak learners *sequentially*, each focused on the
running ensemble's mistakes, combined into an additive model); Chapter 04 (decision trees — here we
use *regression* trees); Module 00 (the train/test split; mean squared error).

**What you'll be able to do.**
- Explain how gradient boosting improves a model by fitting its residuals.
- Build the boosting loop by hand and watch the error fall.
- Reproduce `GradientBoostingRegressor` exactly, to machine precision.
- Read a residual plot and an error-vs-trees curve.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()

SEED = 0
NU = 0.3        # learning rate: the fraction of each tree we add (a shrunken slice)
N_TREES = 100   # number of boosting rounds
MAX_DEPTH = 2   # each weak learner is a shallow regression tree

## Where we are, and one new footing

In Chapter 07, **AdaBoost** trained a sequence of weak learners. After each one it **reweighted the
training points** — making the misclassified ones heavier — so the next learner would focus on them.
The final model was a weighted vote.

Gradient boosting keeps the sequential, focus-on-mistakes spirit but changes the mechanism. Instead
of reweighting points, it **fits the next tree directly to the residuals** — the gap between the
target and the current prediction.

**A new footing: regression.** Every method so far has predicted a *class*. Here, for the first
time, we predict a *number* — the target `y` is continuous. Three words we will use:

- a **prediction** is a number `F(x)`;
- the **residual** is what is left over, `r = y − F(x)`;
- we measure quality with **mean squared error (MSE)**, the average of `r²`.

One more fact we will lean on. A **regression tree** predicts, in each leaf, the **mean of the
training targets** that fall there — the single constant that makes the squared error in that leaf
as small as possible. (Chapter 04's trees voted a class in each leaf; a regression tree puts a
number there instead. Same splitting, numeric leaves.)

## The idea: fit the leftovers

Suppose our first model is the most basic one possible: a flat line at the **mean** of `y`. It is
wrong almost everywhere, and the pattern of how it is wrong — the residuals — still carries all the
structure of the data.

So we fit a small tree to those residuals. It cannot capture everything (it is shallow, deliberately
weak), but it captures *some* of the leftover structure. We add a **fraction** of its prediction to
the model — a small, careful step we will write as `ν` (nu). Now the residuals are smaller, but they
still have structure, so we fit another tree to *them*. And so on.

Each tree is a weak learner that nudges the model toward the data by cleaning up the previous
residuals. AdaBoost reweighted the hard points; gradient boosting fits the leftovers directly.

In [ ]:
# A 1-D regression problem: a smooth target (a sine) seen through noise.
rng = np.random.default_rng(SEED)
n = 120
x = np.sort(rng.uniform(0.0, 2.0 * np.pi, n))
y = np.sin(x) + rng.normal(0.0, 0.25, n)
data = pd.DataFrame({"x": x, "y": y})
X = x.reshape(-1, 1)  # scikit-learn wants a 2-D feature matrix

# The most basic model: a constant, the mean of y.
F0 = y.mean()
mse0 = mean_squared_error(y, np.full(n, F0))
print(f"n = {n} points;  target y = sin(x) + noise")
print(f"F0 = mean(y) = {F0:.4f}   ->   train MSE = {mse0:.4f}")

grid = np.linspace(0.0, 2.0 * np.pi, 400)
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(data["x"], data["y"], color=COLORS["train"], edgecolor=COLORS["text"],
           linewidth=0.5, alpha=0.9, label="data")
ax.plot(grid, np.sin(grid), color=COLORS["muted"], linewidth=1.8, linestyle=":",
        label="true sin(x)")
ax.axhline(F0, color=COLORS["model"], linewidth=2.2, label=f"F0 = mean = {F0:.2f}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

**Read the figure.** The flat blue line is our starting model `F₀` — the mean of `y`. It is a
poor fit: the points rise and fall around it in a clear wave. That wave is exactly the **residual**
`y − F₀`, and it is what the first tree will try to capture. The starting error is MSE ≈ 0.50.

## Round 1: fit a tree to the residuals

Take the residuals `r = y − F₀`, fit a shallow regression tree to them, and add a fraction `ν = 0.3`
of its prediction:

$$F_1(x) = F_0 + \nu \cdot \mathrm{tree}_1(x).$$

We add only a *slice* of the tree (30%), not the whole thing. Taking small steps is what makes
boosting stable; *why* a fraction works better than the full step is the subject of Notebook 4. For
now, we take careful steps.

In [ ]:
# Round 1: fit a shallow regression tree to the residuals of F0, add a shrunken slice.
residual0 = y - F0
tree1 = DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=SEED).fit(X, residual0)
F1 = F0 + NU * tree1.predict(X)

print(f"residuals of F0:  std = {residual0.std():.4f}")
print(f"after round 1:    train MSE  {mse0:.4f}  ->  {mean_squared_error(y, F1):.4f}")

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))
order = np.argsort(x)

# Left: the residuals F0 leaves behind, and the step tree1 fits to them.
axL.scatter(x, residual0, color=COLORS["error"], edgecolor=COLORS["text"],
            linewidth=0.5, alpha=0.85, label="residuals  y − F0")
axL.plot(x[order], tree1.predict(X)[order], color=COLORS["model"], linewidth=2.6,
         label="tree1 fit to residuals")
axL.axhline(0.0, color=COLORS["muted"], linewidth=1.0, linestyle=":")
axL.set_xlabel("x")
axL.set_ylabel("residual")
axL.set_title("what is left over after F0")
axL.legend()

# Right: the updated model F1 = F0 + nu * tree1, over the data.
F1_curve = F0 + NU * tree1.predict(grid.reshape(-1, 1))
axR.scatter(x, y, color=COLORS["train"], edgecolor=COLORS["text"], linewidth=0.5,
            alpha=0.9, label="data")
axR.axhline(F0, color=COLORS["muted"], linewidth=1.5, linestyle="--", label="F0 (mean)")
axR.plot(grid, F1_curve, color=COLORS["highlight"], linewidth=2.6, label="F1 = F0 + 0.3·tree1")
axR.set_xlabel("x")
axR.set_ylabel("y")
axR.set_title("the model after one round")
axR.legend()
plt.show()

**Read the figure.** *Left:* the residuals after `F₀` (coral points) carry a clear shape —
negative where the sine dips, positive where it rises. The blue line is the depth-2 tree's attempt
to capture that shape: a few flat pieces, a coarse but honest summary of where the leftovers are
positive or negative. *Right:* adding `ν = 0.3` of that step to the flat mean gives `F₁` (pink) — no
longer flat, now bending toward the data. One weak tree has pulled the MSE from 0.50 down to 0.30.

## Repeat: each tree cleans up the last one's residuals

Round 1 lowered the error, but `F₁` still misses — there are new, smaller residuals `y − F₁`. So we
do it again: fit a tree to *those*, add a slice, and continue. The model after `m` rounds is

$$F_m(x) = F_{m-1}(x) + \nu \cdot \mathrm{tree}_m(x), \qquad \mathrm{tree}_m \text{ fit on }
r = y - F_{m-1}.$$

Let us run the loop for 100 rounds and watch the error.

In [ ]:
# The full loop: each round fits a tree to the CURRENT residuals, adds a shrunken slice.
F = np.full(n, F0)
trees, stages = [], [F.copy()]
mse_curve = [mean_squared_error(y, F)]
for m in range(N_TREES):
    residual = y - F
    tree = DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=SEED).fit(X, residual)
    F = F + NU * tree.predict(X)
    trees.append(tree)
    stages.append(F.copy())
    mse_curve.append(mean_squared_error(y, F))
mse_curve = np.array(mse_curve)

# One shallow tree fit straight to y, for comparison (a single weak learner alone).
single_tree = DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=SEED).fit(X, y)
mse_single = mean_squared_error(y, single_tree.predict(X))

for k in (0, 1, 2, 3, 4, 5, 10, 20, 50, 100):
    print(f"after {k:3d} trees:  train MSE = {mse_curve[k]:.4f}")
print(f"\nsingle depth-{MAX_DEPTH} tree alone:  train MSE = {mse_single:.4f}")

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))
grid2 = np.linspace(0.0, 2.0 * np.pi, 400)
Xg = grid2.reshape(-1, 1)

# Left: replay the stored trees on a dense grid, snapshotting a few stages.
snaps = [0, 1, 3, 10, 60]
snap_colors = ["muted", "class_d", "class_c", "class_b", "highlight"]
G = np.full(len(grid2), F0)
grid_stages = {0: G.copy()}
for i, tree in enumerate(trees, start=1):
    G = G + NU * tree.predict(Xg)
    if i in snaps:
        grid_stages[i] = G.copy()

axL.scatter(x, y, color=COLORS["train"], edgecolor=COLORS["text"], linewidth=0.5,
            alpha=0.7, label="data", zorder=1)
for key, m in zip(snap_colors, snaps, strict=True):
    axL.plot(grid2, grid_stages[m], color=COLORS[key], linewidth=2.2, zorder=2,
             label=f"F{m}")
axL.set_xlabel("x")
axL.set_ylabel("y")
axL.set_title("the fit builds up, tree by tree")
axL.legend(ncol=2)

# Right: train MSE vs number of trees, with the single-tree reference.
axR.plot(np.arange(len(mse_curve)), mse_curve, color=COLORS["model"], linewidth=2.4,
         label="boosted ensemble")
axR.axhline(mse_single, color=COLORS["error"], linewidth=1.8, linestyle="--",
            label=f"single depth-{MAX_DEPTH} tree")
axR.set_xlabel("number of trees")
axR.set_ylabel("train MSE")
axR.set_title("more trees, smaller error")
axR.legend()
plt.show()

**Read the figure.** *Left:* the model after `m` trees, drawn over the data. `F₀` is the flat
mean; by `F₁₀` the curve already follows the broad wave; by `F₆₀` it traces the sine closely. The
fit is built up in small steps, each tree adding a little more shape. *Right:* the training MSE falls
steeply at first, then more slowly, as trees accumulate. The dashed coral line is a **single** depth-2
tree fit straight to the data (MSE ≈ 0.105). One shallow tree alone cannot do much — but a sum of
shrunken shallow trees passes it by round 4 and keeps improving. Weak learners, stacked on each
other's leftovers, become a strong one. (This is *training* error; that it slips below the noise we
added to the data is a first hint that more trees is not always better — exactly the question
Notebook 4 takes up.)

## Did we get it right? Compare to the library

We built this loop with our own hands: start at the mean, fit a tree to the residuals, add a shrunken
slice, repeat. scikit-learn's `GradientBoostingRegressor` does exactly this. If we match its
settings, our model and its model should agree — the by-hand ↔ library contract the course holds to.

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=N_TREES, learning_rate=NU, max_depth=MAX_DEPTH,
    loss="squared_error", subsample=1.0, random_state=SEED,
).fit(X, y)
F_sklearn = gbr.predict(X)

final_gap = np.max(np.abs(F - F_sklearn))
staged = list(gbr.staged_predict(X))
staged_gap = max(np.max(np.abs(stages[k + 1] - staged[k])) for k in range(N_TREES))

init_const = float(np.ravel(gbr.init_.constant_)[0])
print(f"sklearn init_ = {type(gbr.init_).__name__}(constant={init_const:.4f})   our F0 = {F0:.4f}")
print(f"final  max|by-hand - sklearn|                 = {final_gap:.2e}")
print(f"staged max|by-hand - sklearn| over {N_TREES} rounds = {staged_gap:.2e}")
print(f"has staged_predict: {hasattr(gbr, 'staged_predict')}   has staged_score: "
      f"{hasattr(gbr, 'staged_score')}")

**By hand equals the library, to machine precision.** The largest difference between our
predictions and `GradientBoostingRegressor`'s is around `1e-16` — the limit of floating-point
arithmetic. There is no approximation here: the library is running the loop we wrote. Its starting
model `init_` (the constant model scikit-learn begins from — here a `DummyRegressor` holding the
mean) is our `F₀`, and you read its stage-by-stage predictions with `staged_predict` (note: unlike `AdaBoostClassifier`, it has no `staged_score`, so we compute the
error over `staged_predict` ourselves).

Notice the word we have not used once: **gradient**. We never needed it — fitting the residuals was
enough. Notebook 2 reveals that the residual is, exactly, the *negative gradient* of the squared-error
loss, and that this whole loop is gradient descent in a space of functions. That is where the name
comes from.

## Your turn

1. **The step size `ν`.** Re-run the loop with `ν = 0.1` and then `ν = 1.0`, keeping everything else
   fixed. How does the MSE-vs-trees curve change — which reaches a low error in fewer trees, and which
   is smoother? (We study this trade-off in Notebook 4.)
2. **The weak learner.** Change `MAX_DEPTH` to 1 (stumps) and to 3. How does the single-tree baseline
   move, and how does the ensemble curve change?
3. **The starting point.** Start from `F₀ = 0` instead of the mean. Show that it costs only an extra
   round or two to catch up, and explain — using the recap's "leaf = mean" idea (the mean is the
   constant with the smallest squared error) — why the mean is the best possible constant start.

## What you built

- You started from the mean and **fit the residuals** with shallow regression trees, adding a shrunken
  slice each round.
- You watched the training MSE fall from 0.50 to under 0.01, and the model grow from a flat line into
  the sine curve.
- You reproduced `GradientBoostingRegressor` **exactly** — by-hand and library agree to machine
  precision.

**Vocabulary you now own:** residual · additive model · shrinkage / learning rate `ν` · weak learner
(a shallow regression tree) · staged prediction · the regression-tree leaf is the mean.

## Going further (optional)

- **Why the mean is `F₀`.** Among all constants `c`, the mean of `y` is the one that minimizes the
  squared error `Σ(yᵢ − c)²`. Starting there gives the boosting loop the best possible head start.
- **Why the leaves needed no extra work.** For squared error, the best value to put in a leaf is the
  mean of the residuals there — which is exactly what a regression tree already puts in its leaves. So
  no leaf re-estimation was needed. Notebook 3 will meet a loss (for classification) where the leaf
  *does* need a correction.
- **The name.** Notebook 2 shows that the residual `y − F` is the negative gradient of the
  squared-error loss `½(y − F)²`. Fitting the residuals is therefore taking a step along the negative
  gradient — gradient descent, carried out in the space of functions (Friedman, 2001).

## References

- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
  (§4.1, least-squares regression boosting.)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.9–10.10. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical
  Learning*, §8.2.3. DOI [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).

*Previous: Chapter 07 — AdaBoost.*  ·  *Next: 02 — The residual was the gradient.*